# Strategy - BTC RS Momemtum

In [ ]:
# ============================================================
# S10 - BTC Relative Strength + Momentum
# ============================================================

import xarray as xr
import numpy as np

import qnt.data as qndata
import qnt.ta as qnta
import qnt.output as qnout

# ============================================================
# Load Data
# ============================================================

data = qndata.cryptodaily_load_data(
    min_date="2015-01-01"
)

# ============================================================
# Strategy
# ============================================================

def strategy(data):

    # --------------------------------------------------------
    # Extract Data
    # --------------------------------------------------------

    close = data.sel(field="close")
    is_liquid = data.sel(field="is_liquid")

    close = xr.where(
        is_liquid == 1,
        close,
        np.nan
    )

    # --------------------------------------------------------
    # BTC Data
    # --------------------------------------------------------

    btc = data.sel(
        field="close",
        asset="BTC"
    )

    # --------------------------------------------------------
    # BTC Regime Filter
    # --------------------------------------------------------

    btc_sma200 = qnta.sma(
        btc,
        200
    )

    bull_market = xr.where(
        btc > btc_sma200,
        1,
        0
    )

    # --------------------------------------------------------
    # Returns
    # --------------------------------------------------------

    ret = close / close.shift(time=1) - 1

    vol20 = qnta.std(
        ret,
        20
    )

    # --------------------------------------------------------
    # Momentum
    # --------------------------------------------------------

    momentum = (
        close /
        close.shift(time=21)
        - 1
    )

    momentum_rank = momentum.rank(
        "asset"
    )

    # --------------------------------------------------------
    # BTC Relative Strength
    # --------------------------------------------------------

    btc_momentum = (
        btc /
        btc.shift(time=21)
        - 1
    )

    btc_rs = (
        momentum
        - btc_momentum
    )

    btc_rs_rank = btc_rs.rank(
        "asset"
    )

    # --------------------------------------------------------
    # Combined Score
    # --------------------------------------------------------

    score = (
        0.50 * btc_rs_rank
        +
        0.50 * momentum_rank
    )

    # --------------------------------------------------------
    # Select Top Assets
    # --------------------------------------------------------

    ranks = score.rank(
        "asset"
    )

    top_assets = xr.where(
        ranks >= 6,
        1,
        0
    )

    weights = score * top_assets

    # --------------------------------------------------------
    # Inverse Volatility Weighting
    # --------------------------------------------------------

    inv_vol = (
        1 /
        (vol20 + 1e-6)
    )

    weights = weights * inv_vol

    # --------------------------------------------------------
    # Normalize
    # --------------------------------------------------------

    weights = weights / (
        weights.sum("asset") + 1e-6
    )

    # --------------------------------------------------------
    # Apply BTC Regime Filter
    # --------------------------------------------------------

    weights = weights * bull_market

    return weights.fillna(0)

# ============================================================
# Generate Weights
# ============================================================

weights = strategy(data)

# ============================================================
# Clean Output
# ============================================================

weights = qnout.clean(
    weights,
    data,
    "crypto_daily_long"
)

# ============================================================
# Submission Output
# ============================================================

qnout.write(weights)

100% (15262844 of 15262844) |############| Elapsed Time: 0:00:00 Time:  0:00:00
Output cleaning...
Fix unique timestamps
Forward filling missing prices...
Check liquidity...
Ok.
Check for missed dates...
Ok.
Check positive positions...
Ok.
Final normalization...
Output cleaning complete.
Write output: /root/fractions.nc.gz

In [ ]:
# ============================================================
# Strategy Statistics
# ============================================================

import qnt.stats as qnstats

stats = qnstats.calc_stat(
    data,
    weights.sel(time=slice("2016-01-01", None))
)

stats_pd = stats.to_pandas()

print(stats_pd.tail())

print("\nFinal Metrics:")

print(
    stats_pd.iloc[-1][[
        "equity",
        "sharpe_ratio",
        "max_drawdown",
        "avg_turnover",
        "avg_holding_time"
    ]]
)

field            equity  relative_return  volatility  underwater  \
time                                                               
2026-06-06  8020.192147              0.0    0.711885    -0.30077   
2026-06-07  8020.192147              0.0    0.711792    -0.30077   
2026-06-08  8020.192147              0.0    0.711699    -0.30077   
2026-06-09  8020.192147              0.0    0.711606    -0.30077   
2026-06-10  8020.192147              0.0    0.711514    -0.30077   

field       max_drawdown  sharpe_ratio  mean_return  bias  instruments  \
time                                                                     
2026-06-06     -0.877903      1.561823     1.111838   0.0         31.0   
2026-06-07     -0.877903      1.561617     1.111546   0.0         31.0   
2026-06-08     -0.877903      1.561411     1.111255   0.0         31.0   
2026-06-09     -0.877903      1.561204     1.110963   0.0         31.0   
2026-06-10     -0.877903      1.560998     1.110672   0.0         31.0   

field       avg_turnover  avg_holding_time  
time                                        
2026-06-06      0.217280           5.93698  
2026-06-07      0.217223           5.93698  
2026-06-08      0.217166           5.93698  
2026-06-09      0.217109           5.93698  
2026-06-10      0.217052           5.93698  

Final Metrics:
field
equity              8020.192147
sharpe_ratio           1.560998
max_drawdown          -0.877903
avg_turnover           0.217052
avg_holding_time       5.936980
Name: 2026-06-10 00:00:00, dtype: float64